# Model Diagnostics: Scenario-Based Analysis

This notebook analyzes a trained Deep CFR model using fixed, realistic poker scenarios.

**Key Info:**
- D0 = HIGHEST rank card (e.g., Ace)
- D1 = Middle rank card
- D2 = LOWEST rank card (e.g., 2)
- Good poker: Keep high cards → D2 should have highest discard probability

## Configuration - Change Model Path Here

In [13]:
# ============================================
# CHANGE THIS TO YOUR MODEL PATH
# ============================================
MODEL_PATH = "../output/models/v1_bondi.pt"

# Optional: Compare with another model
COMPARE_MODEL_PATH = None  # Set to path like "../output/models/other_model.pt" to compare
# ============================================

## Setup

In [14]:
import sys
import os
import torch
import torch.nn.functional as F
import pandas as pd
from IPython.display import display, HTML

# Add paths
sys.path.insert(0, '.')
sys.path.insert(0, '..')

from network.model import DeepCFRModule
from utils.infoset_parser import parse_infoset_to_network_input

ACTION_NAMES = ['DISCARD_0', 'DISCARD_1', 'DISCARD_2', 'CHECK', 'CALL', 'FOLD', 'RAISE_S', 'RAISE_M', 'RAISE_L']
print("Setup complete!")

Setup complete!


## Define Scenarios

In [15]:
# Hardcoded realistic scenarios for each game phase
# Format: "S{street}|H:{hand}|B:{board}|A:{history}"
# Hand sorted descending by rank: D0=highest, D2=lowest

SCENARIOS = {
    'PREFLOP BETTING': {
        'legal_indices': [3, 4, 5, 6, 7, 8],
        'scenarios': [
            ('Premium pair (AA-x)', 'S0|H:14s0,14s1,8s2|B:|A:'),
            ('High pair (KK-x)', 'S0|H:13s0,13s1,5s2|B:|A:'),
            ('Broadway (AKQ)', 'S0|H:14s0,13s0,12s0|B:|A:'),
            ('Medium pair (99-x)', 'S0|H:9s0,9s1,6s2|B:|A:'),
            ('Connected (JT9)', 'S0|H:11s0,10s0,9s0|B:|A:'),
            ('One high (A-7-3)', 'S0|H:14s0,7s1,3s2|B:|A:'),
            ('Low cards (7-5-3)', 'S0|H:7s0,5s1,3s2|B:|A:'),
            ('Garbage (8-4-2)', 'S0|H:8s0,4s1,2s2|B:|A:'),
            ('Premium facing raise', 'S0|H:14s0,14s1,10s2|B:|A:R'),
            ('Weak facing raise', 'S0|H:7s0,5s1,2s2|B:|A:R'),
        ]
    },
    
    'DISCARD ROUND': {
        'legal_indices': [0, 1, 2],
        'scenarios': [
            ('Clear low (A-K-3, board 9-7)', 'S0|H:14s0,13s0,3s1|B:9s2,7s2|A:CC'),
            ('Clear low (Q-J-2, board T-8)', 'S0|H:12s0,11s0,2s1|B:10s2,8s2|A:CC'),
            ('Pair + kicker (K-K-5, board A-9)', 'S0|H:13s0,13s1,5s2|B:14s2,9s2|A:CC'),
            ('Pair + kicker (9-9-3, board J-7)', 'S0|H:9s0,9s1,3s2|B:11s2,7s2|A:CC'),
            ('Flush draw (A-T-4 suited, board K-7)', 'S0|H:14s0,10s0,4s0|B:13s0,7s1|A:CC'),
            ('Straight draw (J-T-5, board 9-8)', 'S0|H:11s0,10s0,5s1|B:9s2,8s2|A:CC'),
            ('Close decision (K-Q-J, board A-9)', 'S0|H:13s0,12s0,11s0|B:14s1,9s1|A:CC'),
            ('Close decision (8-7-6, board T-5)', 'S0|H:8s0,7s0,6s0|B:10s1,5s1|A:CC'),
            ('After opp discard', 'S0|H:14s0,13s0,2s1|B:9s2,7s2,12s3|A:CCD'),
            ('After opp discard low', 'S0|H:10s0,8s0,3s1|B:14s2,9s2,5s3|A:CCD'),
        ]
    },
    
    'FLOP BETTING': {
        'legal_indices': [3, 4, 5, 6, 7, 8],
        'scenarios': [
            ('Top pair (K-Q, board K-9-7-5)', 'S1|H:13s0,12s0|B:13s1,9s2,7s2,5s2|A:CCDD'),
            ('Two pair (K-9, board K-9-7-5)', 'S1|H:13s0,9s0|B:13s1,9s1,7s2,5s2|A:CCDD'),
            ('Set (9-9, board K-9-7-5)', 'S1|H:9s0,9s1|B:13s2,9s2,7s2,5s2|A:CCDD'),
            ('Flush draw', 'S1|H:14s0,10s0|B:13s0,7s0,9s1,5s1|A:CCDD'),
            ('Open-ended (J-T, board 9-8-K-3)', 'S1|H:11s0,10s0|B:9s1,8s1,13s2,3s2|A:CCDD'),
            ('Missed (A-Q, board K-9-7-5)', 'S1|H:14s0,12s0|B:13s1,9s2,7s2,5s2|A:CCDD'),
            ('Bottom pair (5-4, board K-9-7-5)', 'S1|H:5s0,4s0|B:13s1,9s2,7s2,5s1|A:CCDD'),
            ('Top pair facing bet', 'S1|H:13s0,12s0|B:13s1,9s2,7s2,5s2|A:CCDDR'),
            ('Draw facing bet', 'S1|H:14s0,10s0|B:13s0,7s0,9s1,5s1|A:CCDDR'),
            ('Air facing bet', 'S1|H:6s0,4s0|B:13s1,9s2,7s2,5s2|A:CCDDR'),
        ]
    },
    
    'TURN BETTING': {
        'legal_indices': [3, 4, 5, 6, 7, 8],
        'scenarios': [
            ('Top pair good kicker', 'S2|H:14s0,13s0|B:14s1,9s2,7s2,5s2,3s2|A:CCDDCC'),
            ('Two pair', 'S2|H:14s0,9s0|B:14s1,9s1,7s2,5s2,3s2|A:CCDDCC'),
            ('Set', 'S2|H:9s0,9s1|B:14s2,9s2,7s2,5s2,3s2|A:CCDDCC'),
            ('Flush draw turn', 'S2|H:14s0,10s0|B:13s0,7s0,9s1,5s1,2s1|A:CCDDCC'),
            ('Missed draw', 'S2|H:11s0,10s0|B:9s1,8s1,14s2,3s2,2s2|A:CCDDCC'),
            ('Second pair', 'S2|H:9s0,8s0|B:14s1,9s1,7s2,5s2,3s2|A:CCDDCC'),
            ('Weak pair', 'S2|H:5s0,4s0|B:14s1,9s2,7s2,5s1,3s2|A:CCDDCC'),
            ('Strong in raised pot', 'S2|H:14s0,14s1|B:14s2,9s2,7s2,5s2,3s2|A:CRDDRCC'),
            ('Draw in raised pot', 'S2|H:14s0,10s0|B:13s0,7s0,9s1,5s1,2s1|A:CRDDRCC'),
            ('Bluff catcher', 'S2|H:9s0,8s0|B:14s1,13s1,12s1,5s2,3s2|A:CCDDCCR'),
        ]
    },
    
    'RIVER BETTING': {
        'legal_indices': [3, 4, 5, 6, 7, 8],
        'scenarios': [
            ('Top pair river', 'S3|H:14s0,13s0|B:14s1,9s2,7s2,5s2,3s2,2s2|A:CCDDCCCC'),
            ('Two pair river', 'S3|H:14s0,9s0|B:14s1,9s1,7s2,5s2,3s2,2s2|A:CCDDCCCC'),
            ('Set river', 'S3|H:9s0,9s1|B:14s2,9s2,7s2,5s2,3s2,2s2|A:CCDDCCCC'),
            ('Straight river', 'S3|H:11s0,10s0|B:9s1,8s1,7s2,5s2,3s2,2s2|A:CCDDCCCC'),
            ('Medium pair vs bet', 'S3|H:9s0,8s0|B:14s1,13s1,12s1,5s2,3s2,2s2|A:CCDDCCCCR'),
            ('Weak pair vs bet', 'S3|H:5s0,4s0|B:14s1,13s1,12s1,9s2,5s1,2s2|A:CCDDCCCCR'),
            ('Missed draw river', 'S3|H:14s0,10s0|B:13s1,7s1,9s2,5s2,3s2,2s2|A:CCDDCCCC'),
            ('Total air river', 'S3|H:6s0,4s0|B:14s1,13s1,12s1,9s2,7s2,2s2|A:CCDDCCCC'),
            ('Nuts in big pot', 'S3|H:14s0,14s1|B:14s2,9s2,7s2,5s2,3s2,2s2|A:CRDDRCRC'),
            ('Bluff catcher big pot', 'S3|H:9s0,9s1|B:14s1,13s1,12s1,10s2,5s2,2s2|A:CRDDRCRCR'),
        ]
    },
}

print(f"Defined {sum(len(p['scenarios']) for p in SCENARIOS.values())} scenarios across {len(SCENARIOS)} phases")

Defined 50 scenarios across 5 phases


## Load Model

In [16]:
def load_model(model_path):
    """Load a model and return network + metadata."""
    print(f"Loading: {model_path}")
    model_data = torch.load(model_path, map_location='cpu', weights_only=False)
    
    network_dim = model_data.get('network_dim', 256)
    iterations = model_data.get('iterations', '?')
    
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=network_dim)
    network.load_state_dict(model_data['strategy_network_state_dict'])
    network.eval()
    
    print(f"  ✓ Loaded! Dim: {network_dim}, Iterations: {iterations}")
    return network, {'dim': network_dim, 'iterations': iterations, 'path': model_path}

# Load main model
network, model_info = load_model(MODEL_PATH)

# Optionally load comparison model
compare_network, compare_info = None, None
if COMPARE_MODEL_PATH:
    compare_network, compare_info = load_model(COMPARE_MODEL_PATH)

Loading: ../output/models/v1_bondi.pt
  ✓ Loaded! Dim: 256, Iterations: 100


## Analysis Functions

In [17]:
def analyze_scenario(network, infoset_str, legal_indices):
    """Get network output for a scenario."""
    try:
        cc, ah = parse_infoset_to_network_input(infoset_str)
        
        with torch.no_grad():
            output = network(cc, ah)
        
        logits = output[0]
        legal_logits = torch.tensor([logits[i].item() for i in legal_indices])
        legal_probs = F.softmax(legal_logits, dim=0)
        
        probs_dict = {}
        for i, idx in enumerate(legal_indices):
            probs_dict[ACTION_NAMES[idx]] = legal_probs[i].item()
        
        dominant_idx = legal_indices[torch.argmax(legal_probs).item()]
        
        return {
            'probs': probs_dict,
            'dominant': ACTION_NAMES[dominant_idx],
            'dominant_prob': torch.max(legal_probs).item(),
        }
    except Exception as e:
        return {'error': str(e)}

def format_probs(probs_dict, highlight_max=True):
    """Format probabilities as a nice string with highlighting."""
    if not probs_dict:
        return "ERROR"
    
    max_key = max(probs_dict.keys(), key=lambda k: probs_dict[k])
    parts = []
    for k, v in probs_dict.items():
        short_name = k.replace('DISCARD_', 'D').replace('CHECK', 'CHK').replace('RAISE_', 'R')
        if highlight_max and k == max_key:
            parts.append(f"**{short_name}: {v:.1%}**")
        else:
            parts.append(f"{short_name}: {v:.1%}")
    return " | ".join(parts)

print("Analysis functions ready!")

Analysis functions ready!


---
# Results by Phase
---

## 1. PREFLOP BETTING

In [18]:
phase = 'PREFLOP BETTING'
phase_data = SCENARIOS[phase]

print(f"═" * 80)
print(f"  {phase}")
print(f"  Legal actions: CHECK, CALL, FOLD, RAISE_S, RAISE_M, RAISE_L")
print(f"═" * 80)
print()

results = []
for desc, infoset in phase_data['scenarios']:
    result = analyze_scenario(network, infoset, phase_data['legal_indices'])
    
    print(f"📋 {desc}")
    print(f"   Infoset: {infoset}")
    if 'error' in result:
        print(f"   ❌ Error: {result['error']}")
    else:
        print(f"   Probs: {format_probs(result['probs'])}")
        print(f"   🎯 Predicted: {result['dominant']} ({result['dominant_prob']:.1%})")
        results.append(result)
    print()

# Average
if results:
    avg = {k: sum(r['probs'][k] for r in results) / len(results) for k in results[0]['probs']}
    print(f"─" * 80)
    print(f"📊 AVERAGE: {format_probs(avg)}")

════════════════════════════════════════════════════════════════════════════════
  PREFLOP BETTING
  Legal actions: CHECK, CALL, FOLD, RAISE_S, RAISE_M, RAISE_L
════════════════════════════════════════════════════════════════════════════════

📋 Premium pair (AA-x)
   Infoset: S0|H:14s0,14s1,8s2|B:|A:
   Probs: CHK: 16.5% | CALL: 18.7% | **FOLD: 19.2%** | RS: 15.4% | RM: 15.0% | RL: 15.1%
   🎯 Predicted: FOLD (19.2%)

📋 High pair (KK-x)
   Infoset: S0|H:13s0,13s1,5s2|B:|A:
   Probs: CHK: 16.6% | CALL: 18.8% | **FOLD: 19.3%** | RS: 15.2% | RM: 15.1% | RL: 15.1%
   🎯 Predicted: FOLD (19.3%)

📋 Broadway (AKQ)
   Infoset: S0|H:14s0,13s0,12s0|B:|A:
   Probs: CHK: 16.6% | CALL: 18.6% | **FOLD: 19.2%** | RS: 15.3% | RM: 15.1% | RL: 15.2%
   🎯 Predicted: FOLD (19.2%)

📋 Medium pair (99-x)
   Infoset: S0|H:9s0,9s1,6s2|B:|A:
   Probs: CHK: 16.5% | CALL: 18.6% | **FOLD: 19.2%** | RS: 15.3% | RM: 15.1% | RL: 15.3%
   🎯 Predicted: FOLD (19.2%)

📋 Connected (JT9)
   Infoset: S0|H:11s0,10s0,9s0|B:|A:


## 2. DISCARD ROUND ⚠️ Critical Phase

In [19]:
phase = 'DISCARD ROUND'
phase_data = SCENARIOS[phase]

print(f"═" * 80)
print(f"  {phase}")
print(f"  D0 = HIGHEST card | D1 = Middle | D2 = LOWEST card")
print(f"  ⚠️ Good strategy: D2 should be highest (discard low cards)")
print(f"═" * 80)
print()

results = []
for desc, infoset in phase_data['scenarios']:
    result = analyze_scenario(network, infoset, phase_data['legal_indices'])
    
    print(f"📋 {desc}")
    print(f"   Infoset: {infoset}")
    if 'error' in result:
        print(f"   ❌ Error: {result['error']}")
    else:
        # Check if prediction makes sense
        is_good = result['dominant'] == 'DISCARD_2'
        status = "✅" if is_good else "⚠️"
        
        print(f"   Probs: {format_probs(result['probs'])}")
        print(f"   {status} Predicted: {result['dominant']} ({result['dominant_prob']:.1%})")
        results.append(result)
    print()

# Average and analysis
if results:
    avg = {k: sum(r['probs'][k] for r in results) / len(results) for k in results[0]['probs']}
    print(f"─" * 80)
    print(f"📊 AVERAGE: {format_probs(avg)}")
    print()
    
    # Analysis
    d0, d1, d2 = avg['DISCARD_0'], avg['DISCARD_1'], avg['DISCARD_2']
    if d2 > d0 and d2 > d1:
        print("✅ GOOD: Model prefers discarding low cards (D2)")
    elif d0 > d1 and d0 > d2:
        print("⚠️ WARNING: Model prefers discarding HIGH cards (D0) - this is backwards!")
    else:
        print("⚠️ UNCLEAR: Model doesn't have clear discard preference yet")

════════════════════════════════════════════════════════════════════════════════
  DISCARD ROUND
  D0 = HIGHEST card | D1 = Middle | D2 = LOWEST card
  ⚠️ Good strategy: D2 should be highest (discard low cards)
════════════════════════════════════════════════════════════════════════════════

📋 Clear low (A-K-3, board 9-7)
   Infoset: S0|H:14s0,13s0,3s1|B:9s2,7s2|A:CC
   Probs: D0: 33.5% | D1: 32.4% | **D2: 34.0%**
   ✅ Predicted: DISCARD_2 (34.0%)

📋 Clear low (Q-J-2, board T-8)
   Infoset: S0|H:12s0,11s0,2s1|B:10s2,8s2|A:CC
   Probs: D0: 33.6% | D1: 32.4% | **D2: 34.0%**
   ✅ Predicted: DISCARD_2 (34.0%)

📋 Pair + kicker (K-K-5, board A-9)
   Infoset: S0|H:13s0,13s1,5s2|B:14s2,9s2|A:CC
   Probs: D0: 33.5% | D1: 32.5% | **D2: 34.0%**
   ✅ Predicted: DISCARD_2 (34.0%)

📋 Pair + kicker (9-9-3, board J-7)
   Infoset: S0|H:9s0,9s1,3s2|B:11s2,7s2|A:CC
   Probs: D0: 33.6% | D1: 32.3% | **D2: 34.0%**
   ✅ Predicted: DISCARD_2 (34.0%)

📋 Flush draw (A-T-4 suited, board K-7)
   Infoset: S0|H:14

## 3. FLOP BETTING

In [20]:
phase = 'FLOP BETTING'
phase_data = SCENARIOS[phase]

print(f"═" * 80)
print(f"  {phase}")
print(f"  Legal actions: CHECK, CALL, FOLD, RAISE_S, RAISE_M, RAISE_L")
print(f"═" * 80)
print()

results = []
for desc, infoset in phase_data['scenarios']:
    result = analyze_scenario(network, infoset, phase_data['legal_indices'])
    
    print(f"📋 {desc}")
    print(f"   Infoset: {infoset}")
    if 'error' in result:
        print(f"   ❌ Error: {result['error']}")
    else:
        print(f"   Probs: {format_probs(result['probs'])}")
        print(f"   🎯 Predicted: {result['dominant']} ({result['dominant_prob']:.1%})")
        results.append(result)
    print()

if results:
    avg = {k: sum(r['probs'][k] for r in results) / len(results) for k in results[0]['probs']}
    print(f"─" * 80)
    print(f"📊 AVERAGE: {format_probs(avg)}")

════════════════════════════════════════════════════════════════════════════════
  FLOP BETTING
  Legal actions: CHECK, CALL, FOLD, RAISE_S, RAISE_M, RAISE_L
════════════════════════════════════════════════════════════════════════════════

📋 Top pair (K-Q, board K-9-7-5)
   Infoset: S1|H:13s0,12s0|B:13s1,9s2,7s2,5s2|A:CCDD
   Probs: **CHK: 19.0%** | CALL: 17.7% | FOLD: 18.1% | RS: 15.7% | RM: 14.8% | RL: 14.7%
   🎯 Predicted: CHECK (19.0%)

📋 Two pair (K-9, board K-9-7-5)
   Infoset: S1|H:13s0,9s0|B:13s1,9s1,7s2,5s2|A:CCDD
   Probs: **CHK: 19.0%** | CALL: 17.6% | FOLD: 18.1% | RS: 15.7% | RM: 14.8% | RL: 14.7%
   🎯 Predicted: CHECK (19.0%)

📋 Set (9-9, board K-9-7-5)
   Infoset: S1|H:9s0,9s1|B:13s2,9s2,7s2,5s2|A:CCDD
   Probs: **CHK: 18.8%** | CALL: 17.8% | FOLD: 18.1% | RS: 15.7% | RM: 14.8% | RL: 14.7%
   🎯 Predicted: CHECK (18.8%)

📋 Flush draw
   Infoset: S1|H:14s0,10s0|B:13s0,7s0,9s1,5s1|A:CCDD
   Probs: **CHK: 19.1%** | CALL: 17.4% | FOLD: 18.1% | RS: 15.8% | RM: 14.8% | RL: 14.8

## 4. TURN BETTING

In [21]:
phase = 'TURN BETTING'
phase_data = SCENARIOS[phase]

print(f"═" * 80)
print(f"  {phase}")
print(f"  Legal actions: CHECK, CALL, FOLD, RAISE_S, RAISE_M, RAISE_L")
print(f"═" * 80)
print()

results = []
for desc, infoset in phase_data['scenarios']:
    result = analyze_scenario(network, infoset, phase_data['legal_indices'])
    
    print(f"📋 {desc}")
    print(f"   Infoset: {infoset}")
    if 'error' in result:
        print(f"   ❌ Error: {result['error']}")
    else:
        print(f"   Probs: {format_probs(result['probs'])}")
        print(f"   🎯 Predicted: {result['dominant']} ({result['dominant_prob']:.1%})")
        results.append(result)
    print()

if results:
    avg = {k: sum(r['probs'][k] for r in results) / len(results) for k in results[0]['probs']}
    print(f"─" * 80)
    print(f"📊 AVERAGE: {format_probs(avg)}")

════════════════════════════════════════════════════════════════════════════════
  TURN BETTING
  Legal actions: CHECK, CALL, FOLD, RAISE_S, RAISE_M, RAISE_L
════════════════════════════════════════════════════════════════════════════════

📋 Top pair good kicker
   Infoset: S2|H:14s0,13s0|B:14s1,9s2,7s2,5s2,3s2|A:CCDDCC
   Probs: CHK: 17.9% | **CALL: 18.9%** | FOLD: 18.3% | RS: 15.4% | RM: 14.8% | RL: 14.7%
   🎯 Predicted: CALL (18.9%)

📋 Two pair
   Infoset: S2|H:14s0,9s0|B:14s1,9s1,7s2,5s2,3s2|A:CCDDCC
   Probs: CHK: 18.2% | **CALL: 18.7%** | FOLD: 18.2% | RS: 15.5% | RM: 14.8% | RL: 14.7%
   🎯 Predicted: CALL (18.7%)

📋 Set
   Infoset: S2|H:9s0,9s1|B:14s2,9s2,7s2,5s2,3s2|A:CCDDCC
   Probs: CHK: 18.0% | **CALL: 18.8%** | FOLD: 18.3% | RS: 15.3% | RM: 14.8% | RL: 14.7%
   🎯 Predicted: CALL (18.8%)

📋 Flush draw turn
   Infoset: S2|H:14s0,10s0|B:13s0,7s0,9s1,5s1,2s1|A:CCDDCC
   Probs: CHK: 18.3% | **CALL: 18.5%** | FOLD: 18.1% | RS: 15.6% | RM: 14.8% | RL: 14.7%
   🎯 Predicted: CALL (1

## 5. RIVER BETTING

In [22]:
phase = 'RIVER BETTING'
phase_data = SCENARIOS[phase]

print(f"═" * 80)
print(f"  {phase}")
print(f"  Legal actions: CHECK, CALL, FOLD, RAISE_S, RAISE_M, RAISE_L")
print(f"═" * 80)
print()

results = []
for desc, infoset in phase_data['scenarios']:
    result = analyze_scenario(network, infoset, phase_data['legal_indices'])
    
    print(f"📋 {desc}")
    print(f"   Infoset: {infoset}")
    if 'error' in result:
        print(f"   ❌ Error: {result['error']}")
    else:
        print(f"   Probs: {format_probs(result['probs'])}")
        print(f"   🎯 Predicted: {result['dominant']} ({result['dominant_prob']:.1%})")
        results.append(result)
    print()

if results:
    avg = {k: sum(r['probs'][k] for r in results) / len(results) for k in results[0]['probs']}
    print(f"─" * 80)
    print(f"📊 AVERAGE: {format_probs(avg)}")

════════════════════════════════════════════════════════════════════════════════
  RIVER BETTING
  Legal actions: CHECK, CALL, FOLD, RAISE_S, RAISE_M, RAISE_L
════════════════════════════════════════════════════════════════════════════════

📋 Top pair river
   Infoset: S3|H:14s0,13s0|B:14s1,9s2,7s2,5s2,3s2,2s2|A:CCDDCCCC
   Probs: CHK: 17.6% | **CALL: 19.1%** | FOLD: 18.3% | RS: 15.5% | RM: 14.8% | RL: 14.7%
   🎯 Predicted: CALL (19.1%)

📋 Two pair river
   Infoset: S3|H:14s0,9s0|B:14s1,9s1,7s2,5s2,3s2,2s2|A:CCDDCCCC
   Probs: CHK: 17.9% | **CALL: 18.9%** | FOLD: 18.2% | RS: 15.5% | RM: 14.8% | RL: 14.7%
   🎯 Predicted: CALL (18.9%)

📋 Set river
   Infoset: S3|H:9s0,9s1|B:14s2,9s2,7s2,5s2,3s2,2s2|A:CCDDCCCC
   Probs: CHK: 17.7% | **CALL: 19.1%** | FOLD: 18.3% | RS: 15.4% | RM: 14.8% | RL: 14.7%
   🎯 Predicted: CALL (19.1%)

📋 Straight river
   Infoset: S3|H:11s0,10s0|B:9s1,8s1,7s2,5s2,3s2,2s2|A:CCDDCCCC
   Probs: CHK: 18.1% | **CALL: 18.7%** | FOLD: 18.1% | RS: 15.7% | RM: 14.7% | RL: 

---
# Summary Table
---

In [23]:
print("═" * 80)
print(f"  SUMMARY: {os.path.basename(MODEL_PATH)} ({model_info['iterations']} iterations)")
print("═" * 80)
print()

summary_data = []

for phase_name, phase_data in SCENARIOS.items():
    results = []
    for desc, infoset in phase_data['scenarios']:
        result = analyze_scenario(network, infoset, phase_data['legal_indices'])
        if 'probs' in result:
            results.append(result)
    
    if results:
        avg = {k: sum(r['probs'][k] for r in results) / len(results) for k in results[0]['probs']}
        dominant = max(avg.keys(), key=lambda k: avg[k])
        
        row = {'Phase': phase_name, 'Dominant': dominant, 'Prob': f"{avg[dominant]:.1%}"}
        for k, v in avg.items():
            row[k] = f"{v:.1%}"
        summary_data.append(row)

# Display as DataFrame
df = pd.DataFrame(summary_data)
display(df)

════════════════════════════════════════════════════════════════════════════════
  SUMMARY: v1_bondi.pt (100 iterations)
════════════════════════════════════════════════════════════════════════════════



,Phase,Dominant,Prob,CHECK,CALL,FOLD,RAISE_S,RAISE_M,RAISE_L,DISCARD_0,DISCARD_1,DISCARD_2
0,PREFLOP BETTING,FOLD,19.2%,16.5%,18.8%,19.2%,15.3%,15.0%,15.1%,NaN,NaN,NaN
1,DISCARD ROUND,DISCARD_2,34.1%,NaN,NaN,NaN,NaN,NaN,NaN,33.6%,32.4%,34.1%
2,FLOP BETTING,CHECK,19.5%,19.5%,17.5%,17.8%,15.8%,14.8%,14.7%,NaN,NaN,NaN
3,TURN BETTING,CHECK,18.8%,18.8%,18.2%,17.9%,15.6%,14.8%,14.7%,NaN,NaN,NaN
4,RIVER BETTING,CHECK,18.7%,18.7%,18.4%,18.0%,15.6%,14.7%,14.7%,NaN,NaN,NaN


---
# Model Comparison (if enabled)
---

In [24]:
if compare_network is not None:
    print("═" * 80)
    print("  MODEL COMPARISON")
    print("═" * 80)
    print(f"  Model 1: {os.path.basename(MODEL_PATH)} ({model_info['iterations']} iters)")
    print(f"  Model 2: {os.path.basename(COMPARE_MODEL_PATH)} ({compare_info['iterations']} iters)")
    print()
    
    comparison_data = []
    
    for phase_name, phase_data in SCENARIOS.items():
        # Model 1
        results1 = [analyze_scenario(network, infoset, phase_data['legal_indices']) 
                    for _, infoset in phase_data['scenarios']]
        results1 = [r for r in results1 if 'probs' in r]
        
        # Model 2
        results2 = [analyze_scenario(compare_network, infoset, phase_data['legal_indices']) 
                    for _, infoset in phase_data['scenarios']]
        results2 = [r for r in results2 if 'probs' in r]
        
        if results1 and results2:
            avg1 = {k: sum(r['probs'][k] for r in results1) / len(results1) for k in results1[0]['probs']}
            avg2 = {k: sum(r['probs'][k] for r in results2) / len(results2) for k in results2[0]['probs']}
            
            dom1 = max(avg1.keys(), key=lambda k: avg1[k])
            dom2 = max(avg2.keys(), key=lambda k: avg2[k])
            
            comparison_data.append({
                'Phase': phase_name,
                'Model1 Dominant': f"{dom1} ({avg1[dom1]:.1%})",
                'Model2 Dominant': f"{dom2} ({avg2[dom2]:.1%})",
            })
    
    df = pd.DataFrame(comparison_data)
    display(df)
else:
    print("ℹ️ Set COMPARE_MODEL_PATH at the top to compare two models")

ℹ️ Set COMPARE_MODEL_PATH at the top to compare two models
